In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile
from pathlib import Path
import copy
import glob 

In [ ]:
path = "/home/pat/Schreibtisch/Dokumente/Uni Freiburg/WS 26 27/GitRep/datascience_env_monit/scripts/data/raw/AMF_US-UiD_FLUXNET_2008-2024_v1.3_r1.zip"


    # [N]_[CC-###]_FLUXNET_FLUXMET_[T]_[YS]-[YE]_[V]_[R].csv

    # where:
    # [N] = code identifying the source network** 
    # [CC-###] = the FLUXNET site code
    # [T] = the time resolution, with options HH for hourly or half-hourly, DD for daily, WW for weekly, MM for monthly, and YY for yearly
    # [YS]-[YE] = the start and end years of the measurements
    # [V] = the version of ONEFlux used*
    # [R] = the data release version



In [23]:

keep = ["TIMESTAMP_START", "TIMESTAMP_END", "TA_F", "SW_IN_F", "VPD_F", "P_F",
        "NETRAD", "G_F_MDS", "H_F_MDS", "H_F_MDS_QC", "LE_F_MDS", "LE_F_MDS_QC",
        "NEE_VUT_REF", "NEE_VUT_REF_QC", "GPP_NT_VUT_REF", "RECO_NT_VUT_REF"]

zip_file = sorted(Path("/home/pat/Schreibtisch/Dokumente/Uni Freiburg/WS 26 27/GitRep/datascience_env_monit/scripts/data/raw").glob("*.zip"))[1]
with zipfile.ZipFile(zip_file) as z:
    hh_name = [n for n in z.namelist() if "FLUXMET_HH" in n][0]
    with z.open(hh_name) as f:
        hh = pd.read_csv(f, na_values=[-9999],
                         usecols=lambda c: c in keep,
                         dtype={"TIMESTAMP_START": str, "TIMESTAMP_END": str})

hh["time"] = pd.to_datetime(hh["TIMESTAMP_START"], format="%Y%m%d%H%M")
print(len(hh), hh["time"].min(), hh["time"].max())
print((hh["NEE_VUT_REF_QC"] == 0).mean())    # share of measured NEE

210384 2013-01-01 00:00:00 2024-12-31 23:30:00
0.28946117575481023


In [24]:
#print(hh.head(-10))

B3 -> Turn B2 into a function read_hh(zip_file) that 
returns one tidy table with a site column (the site ID from 
the file name), then read all five towers into one table. 
You will reuse this function in Block 5.

In [46]:

def readzip(folder, keep):
    hh = pd.DataFrame()
    zip_files = sorted(Path(folder).glob("*.zip"))

    for zip_file in zip_files:
        with zipfile.ZipFile(zip_file) as z:
            hh_names = [
                name for name in z.namelist()
                if "FLUXMET_HH" in name
            ]

            for hh_name in hh_names:
                with z.open(hh_name) as f:
                    current = pd.read_csv(
                        f,
                        na_values=[-9999],
                        usecols=lambda c: c in keep,
                        dtype={
                            "TIMESTAMP_START": str,
                            "TIMESTAMP_END": str,
                        },
                    )

                current["time"] = pd.to_datetime(
                    current["TIMESTAMP_START"],
                    format="%Y%m%d%H%M",
                )
                current["site"] = hh_name.split("_FLUXNET_FLUXMET_HH")[0]

                hh = pd.concat([hh, current], ignore_index=True)

    return hh

hh_df = readzip(path,keep)

In [48]:
print(hh_df['site'].unique())
print(hh_df.head())


<ArrowStringArray>
['AMF_US-UiD', 'CNF_CN-Sdq', 'EUF_RU-Fyo', 'ICOS_DE-Hai', 'ICOS_FR-Gri']
Length: 5, dtype: str
  TIMESTAMP_START TIMESTAMP_END   TA_F  SW_IN_F  VPD_F    P_F  NETRAD  \
0    200801010000  200801010030 -5.472      0.0  0.477  0.034     NaN   
1    200801010030  200801010100 -5.525      0.0  0.500  0.034     NaN   
2    200801010100  200801010130 -5.578      0.0  0.524  0.015     NaN   
3    200801010130  200801010200 -5.608      0.0  0.524  0.015     NaN   
4    200801010200  200801010230 -5.638      0.0  0.523  0.013     NaN   

   G_F_MDS  LE_F_MDS  LE_F_MDS_QC  H_F_MDS  H_F_MDS_QC  RECO_NT_VUT_REF  \
0      NaN       NaN          NaN      NaN         NaN              NaN   
1      NaN       NaN          NaN      NaN         NaN              NaN   
2      NaN       NaN          NaN      NaN         NaN              NaN   
3      NaN       NaN          NaN      NaN         NaN              NaN   
4      NaN       NaN          NaN      NaN         NaN              NaN 

## Part C

Create reports/block4_data_summary.qmd (or .ipynb) in your fork. It must render to HTML from a fresh clone with the commands in your README, provided the data are in data/raw.